# Final Project: Rama Llama's GRU Ensemble (OPIM 5509)
**Leaderboard nickname: Rama llama** · NetID raa20002

This notebook trains, checks and ships my final project model: an **ensemble of 5 GRU encoder-decoder networks**,
averaged inside one `model.keras`, that gives the 10th, 50th and 90th percentile of Connecticut's load for each of the
next 24 hours. It builds on FP1/FP2 with five changes, each tested on 2025 (validation) before the one look at 2026:

| Change | Why |
| :-- | :-- |
| **Trained on HRRR weather forecasts as issued** (the run that was really available ~7 hours after it started), not the weather that happened | live, the model only ever sees a forecast; training on forecasts teaches it how much to trust them |
| **Sunlight (`solar_wm2`) from HRRR** as a 6th weather column | rooftop solar hides load at midday; this was worth ~3 MW of CRPS on 2025 |
| **Encoder-decoder**: Conv1D + GRU read the past week, a second GRU walks the next 24 hours | each forecast hour gets its own weather and calendar row, instead of one long 192-row table |
| **Load anchors** in every future row: same hour yesterday, same hour last week, the newest hour | the model can copy yesterday's shape on calm days (when "same hour yesterday" is hard to beat) |
| **5 networks averaged, trained on every hour through Oct 2026** | averaging cuts the noise of any one network; the newest data carries 2026's load shape (more solar, more electric heat) |

The range comes from the pinball loss (the CRPS is the sum of the three pinball losses ÷ 1.5) and one calibration knob,
`range_widen`, picked on 2025. **Runtime > Change runtime type > T4 GPU** (about 15 minutes in all).

In [ ]:
import io, json, time
import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import layers
from pandas.tseries.holiday import USFederalHolidayCalendar

SETTINGS = {
    "seed": 5509,
    "lookback_hours": 168,       # the past week of load
    "horizon_hours": 24,         # the next 24 hours
    "hrrr_lag_hours": 7,         # an HRRR run reaches the class file about 7 hours after it starts
    "units": 128,                # GRU size
    "conv_filters": 32,
    "members": 5,                # networks in the ensemble
    "epochs": 12,                # fixed (early stopping on 2025 picked 9-19; 12 for every member)
    "batch_size": 256,
}
QUANTILES = [0.1, 0.5, 0.9]
print("Keras", keras.__version__, "| GPU:", len(tf.config.list_physical_devices("GPU")) > 0)

## Part 1 · The data
The same hourly load file as FP1/FP2 (2023 to Oct 2026), and every **HRRR forecast for Bradley as it was issued**
(4 runs a day, 48 hours ahead each) from the OPIM 5512 data folder: the same kind of forecast the class HRRR file gives
`forecast-24h-dl` every hour.

In [ ]:
BASE = "https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/Module4/data/"
df = pd.read_csv(BASE + "bdl_ct_load_hourly_2023_2026.csv", parse_dates=["hour_utc"]).set_index("hour_utc").sort_index()
df = df.reindex(pd.date_range(df.index.min(), df.index.max(), freq="h"))
hrrr = pd.read_csv(BASE + "bdl_hrrr_asissued_2023_2026.csv.gz", parse_dates=["init_utc", "hour_utc"])
WEATHER_COLUMNS = ["temp_f", "dewpoint_f", "rel_humidity_pct", "wind_speed_mph", "cloud_cover_pct", "solar_wm2"]
print("load:", df.shape, df.index.min(), "to", df.index.max())
print("HRRR runs:", hrrr["init_utc"].nunique(), "from", hrrr["init_utc"].min(), "to", hrrr["init_utc"].max())

## Part 2 · The score: CRPS and CRPS skill (same as the leaderboard)

In [ ]:
def crps(actual, low, expected, high):
    width = high - low
    outside = np.clip(low - actual, 0, None) + np.clip(actual - high, 0, None)
    return (0.5 * np.abs(actual - expected) + 0.1 * width + outside) / 1.5


def crps_skill(scores, baseline_error):
    skills = [1 - np.nanmean(scores[:, a:b]) / np.nanmean(baseline_error[:, a:b]) for a, b in [(0, 6), (6, 12), (12, 18), (18, 24)]]
    return float(np.mean(skills))

## Part 3 · The inputs, built exactly like the cloud
**`make_inputs` is the contract with `forecast-24h-dl`** (my fork has a word-for-word copy, `make_inputs_rama_v1`):

* **past** `(168, 5)`: the scaled load for the past week + hour of day and day of week (sine, cosine)
* **future** `(24, 17)`: for each of the next 24 hours, the 6 HRRR weather columns (scaled), the 8 calendar columns, and
  3 **load anchors**: the same hour yesterday (2 days back for hour 24, whose "yesterday" is the hour still in progress),
  the same hour last week, and the newest finished hour.

For 60,000 training samples it is faster to build the same numbers for every hour at once (`build_samples`); the check
below proves both give identical inputs. Each issue hour T uses the **newest HRRR run that existed** (started at least 7
hours before T), and, as extra training data, the run before it.

In [ ]:
HOLIDAYS = USFederalHolidayCalendar().holidays(start="2020-01-01", end="2030-12-31")
LB, H = SETTINGS["lookback_hours"], SETTINGS["horizon_hours"]


def calendar_features(hours_utc):
    """Hour of day, day of week and time of year as sines and cosines, plus weekend and holiday flags, in Connecticut time."""
    local = pd.DatetimeIndex(hours_utc).tz_convert("America/New_York")
    out = pd.DataFrame(index=pd.DatetimeIndex(hours_utc))
    out["hour_sin"] = np.sin(2 * np.pi * local.hour / 24)
    out["hour_cos"] = np.cos(2 * np.pi * local.hour / 24)
    out["dow_sin"] = np.sin(2 * np.pi * local.dayofweek / 7)
    out["dow_cos"] = np.cos(2 * np.pi * local.dayofweek / 7)
    out["doy_sin"] = np.sin(2 * np.pi * local.dayofyear / 365.25)
    out["doy_cos"] = np.cos(2 * np.pi * local.dayofyear / 365.25)
    out["is_weekend"] = (local.dayofweek >= 5).astype(float)
    out["is_holiday"] = local.tz_localize(None).normalize().isin(HOLIDAYS).astype(float)
    return out


def make_inputs(issue_hour_utc, load, weather, card):
    """THE CONTRACT (copied word for word in forecast-24h-dl as make_inputs_rama_v1). Two inputs for ONE forecast."""
    issue = pd.Timestamp(issue_hour_utc)
    lookback = int(card["lookback_hours"])
    horizon = int(card["horizon_hours"])
    past_hours = pd.date_range(issue - pd.Timedelta(hours=lookback), periods=lookback, freq="h")
    future_hours = pd.date_range(issue + pd.Timedelta(hours=1), periods=horizon, freq="h")
    past_load = (load.reindex(past_hours).values - card["load_mean"]) / card["load_std"]
    past_cal = calendar_features(past_hours)[["hour_sin", "hour_cos", "dow_sin", "dow_cos"]].values
    past = np.column_stack([past_load, past_cal])
    columns = []
    for column in card["weather_columns"]:
        values = weather[column].reindex(future_hours).values
        columns.append((values - card["weather_mean"][column]) / card["weather_std"][column])
    future = np.column_stack(columns + [calendar_features(future_hours).values])
    # position j in past_load is hour T-168+j: the same hour yesterday of T+h is j = 144+h (h = 1..23), T+24 -> j = 144
    yesterday = past_load[np.r_[144 + np.arange(1, 24), 144]]
    last_week = past_load[np.arange(1, 25)]
    newest = np.repeat(past_load[-1], 24)
    future = np.column_stack([future, yesterday, last_week, newest])
    return past.astype("float32"), future.astype("float32")

In [ ]:
def build_samples(run_offset=0):
    """Every issue hour T at once: the past week of load (MW), the HRRR run that was newest at T (run_offset=1: the
    one before), the calendar, and the answer y (the next 24 hours of load, MW)."""
    hours = df.index
    load = df["load_mw"].values.astype("float64")
    inits = np.array(sorted(hrrr["init_utc"].dt.tz_convert(None).unique())).astype("datetime64[ns]")
    index_of = {pd.Timestamp(t).tz_localize("UTC"): i for i, t in enumerate(inits)}
    W = np.full((len(inits), 49, len(WEATHER_COLUMNS)), np.nan)          # run x lead x weather column
    W[hrrr["init_utc"].map(index_of).values, hrrr["lead_hours"].values] = hrrr[WEATHER_COLUMNS].values
    cal = calendar_features(hours).values.astype("float32")
    T_idx = np.arange(LB, len(hours) - H)
    T = hours[T_idx]
    available = (T - pd.Timedelta(hours=SETTINGS["hrrr_lag_hours"])).tz_convert(None).values.astype("datetime64[ns]")
    k = np.searchsorted(inits, available, side="right") - 1 - run_offset
    keep = k >= 0
    T_idx, T, k = T_idx[keep], T[keep], k[keep]
    lead0 = ((T.tz_convert(None).values.astype("datetime64[ns]") - inits[k]) / np.timedelta64(1, "h")).astype(int) + 1
    leads = lead0[:, None] + np.arange(H)[None, :]
    keep = leads[:, -1] <= 48
    T_idx, T, k, leads = T_idx[keep], T[keep], k[keep], leads[keep]
    fw = W[k[:, None], leads]
    past = np.stack([load[T_idx - LB + j] for j in range(LB)], 1)
    fut_i = T_idx[:, None] + 1 + np.arange(H)[None, :]
    y = load[fut_i]
    good = ~(np.isnan(past).any(1) | np.isnan(y).any(1) | np.isnan(fw).any((1, 2)))
    return {"T": T[good], "past": past[good], "y": y[good], "fw": fw[good], "init": inits[k][good],
            "past_cal": cal[T_idx[:, None] - LB + np.arange(LB)[None, :]][good], "fut_cal": cal[fut_i][good]}


start = time.time()
newest_run = build_samples(0)
previous_run = build_samples(1)
print(f"built in {time.time() - start:.0f} s:", len(newest_run["T"]), "issue hours (+", len(previous_run["T"]), "with the run before)")

In [ ]:
def make_card(train_end):
    """Scaling numbers from the training hours only."""
    train_hours = df.index < train_end
    first = newest_run["fw"][(newest_run["T"] + pd.Timedelta(hours=24)) < train_end].reshape(-1, len(WEATHER_COLUMNS))
    return {"model_type": "keras_gru_ensemble", "feature_set": "rama_v1", "lookback_hours": LB, "horizon_hours": H,
            "weather_columns": WEATHER_COLUMNS,
            "load_mean": float(df.loc[train_hours, "load_mw"].mean()), "load_std": float(df.loc[train_hours, "load_mw"].std()),
            "weather_mean": {c: float(first[:, i].mean()) for i, c in enumerate(WEATHER_COLUMNS)},
            "weather_std": {c: float(first[:, i].std()) for i, c in enumerate(WEATHER_COLUMNS)}}


def features(d, card):
    """make_inputs for every sample at once (fast)."""
    p = (d["past"] - card["load_mean"]) / card["load_std"]
    wm = np.array([card["weather_mean"][c] for c in WEATHER_COLUMNS]); ws = np.array([card["weather_std"][c] for c in WEATHER_COLUMNS])
    past = np.concatenate([p[:, :, None], d["past_cal"][:, :, :4]], -1)
    j24 = np.r_[144 + np.arange(1, 24), 144]
    fut = np.concatenate([(d["fw"] - wm) / ws, d["fut_cal"], p[:, j24][:, :, None], p[:, 1:25][:, :, None],
                          np.repeat(p[:, -1:, None], 24, 1)], -1)
    return past.astype("float32"), fut.astype("float32"), ((d["y"] - card["load_mean"]) / card["load_std"]).astype("float32")


def subset(d, first_issue, last_target):
    m = (d["T"] >= first_issue) & ((d["T"] + pd.Timedelta(hours=24)) < last_target)
    return {k: v[m] for k, v in d.items()}


# ---- The check: make_inputs (the cloud's code) == features (the fast training code), on 8 random samples ----
check_card = make_card(pd.Timestamp("2025-01-01", tz="UTC"))
P, F, _ = features(newest_run, check_card)
rng = np.random.default_rng(SETTINGS["seed"])
for i in rng.choice(len(newest_run["T"]), 8, replace=False):
    run = hrrr[hrrr["init_utc"] == pd.Timestamp(newest_run["init"][i]).tz_localize("UTC")].set_index("hour_utc")
    past, future = make_inputs(newest_run["T"][i], df["load_mw"], run, check_card)
    if not (np.allclose(past, P[i], atol=1e-5) and np.allclose(future, F[i], atol=1e-5)):
        raise ValueError("make_inputs does not match features at " + str(newest_run["T"][i]))
print("8 random samples: make_inputs == features | past", past.shape, "future", future.shape)

## Part 4 · The network
One **member**: `Conv1D` (32 filters, 3 hours wide) + `MaxPooling1D` + `GRU(128)` read the past week into one summary;
`RepeatVector` hands that summary to each of the next 24 hours, next to that hour's weather, calendar and load anchors;
a second `GRU(128)` walks the 24 hours in order; `TimeDistributed(Dense)` layers give 3 numbers per hour (10th, 50th,
90th percentile). A linear **skip** (`TimeDistributed(Dense)` straight from the future inputs, added at the end) lets a
member pass the anchors through almost untouched when that's the best guess. The loss is the pinball loss.
The **ensemble**: 5 members with different random starts, averaged by an `Average` layer inside one `keras.Model`.

In [ ]:
Q = np.array(QUANTILES, dtype="float32")


def pinball_loss(y_true, y_pred):
    error = keras.ops.expand_dims(y_true, -1) - y_pred
    return keras.ops.mean(keras.ops.maximum(Q * error, (Q - 1) * error))


def member(i, past_in, future_in):
    x = layers.Conv1D(SETTINGS["conv_filters"], 3, padding="same", activation="relu", name=f"m{i}_conv")(past_in)
    x = layers.MaxPooling1D(2, name=f"m{i}_pool")(x)
    summary = layers.GRU(SETTINGS["units"], name=f"m{i}_gru_past")(x)
    z = layers.Concatenate(name=f"m{i}_concat")([future_in, layers.RepeatVector(H, name=f"m{i}_repeat")(summary)])
    z = layers.GRU(SETTINGS["units"], return_sequences=True, name=f"m{i}_gru_future")(z)
    z = layers.TimeDistributed(layers.Dense(32, activation="relu"), name=f"m{i}_dense")(z)
    out = layers.TimeDistributed(layers.Dense(len(QUANTILES)), name=f"m{i}_quantiles")(z)
    skip = layers.TimeDistributed(layers.Dense(len(QUANTILES)), name=f"m{i}_skip")(future_in)
    return layers.Add(name=f"m{i}_add")([out, skip])


def train_ensemble(train_end, members=SETTINGS["members"], verbose=True):
    """Train on every issue hour whose 24 targets end before train_end (newest run + the run before)."""
    card = make_card(train_end)
    parts = [features(subset(d, pd.Timestamp("2000-01-01", tz="UTC"), train_end), card) for d in (newest_run, previous_run)]
    Xp = np.concatenate([p[0] for p in parts]); Xf = np.concatenate([p[1] for p in parts]); Y = np.concatenate([p[2] for p in parts])
    past_in = keras.Input((LB, Xp.shape[2]), name="past")
    future_in = keras.Input((H, Xf.shape[2]), name="future")
    outs = []
    for i in range(members):
        keras.utils.set_random_seed(SETTINGS["seed"] + i)
        out = member(i, past_in, future_in)
        net = keras.Model([past_in, future_in], out, name=f"net{i}")
        net.compile(optimizer="adam", loss=pinball_loss)
        start = time.time()
        net.fit([Xp, Xf], Y, epochs=SETTINGS["epochs"], batch_size=SETTINGS["batch_size"], verbose=0, shuffle=True)
        if verbose:
            print(f"  member {i + 1}/{members}: {time.time() - start:.0f} s, {len(Y)} samples")
        outs.append(out)
    averaged = layers.Average(name="ensemble_average")(outs) if len(outs) > 1 else outs[0]
    ensemble = keras.Model([past_in, future_in], averaged, name="rama_llama_ensemble")
    return ensemble, card


def forecast(model, card, d, widen=1.0):
    P, F, _ = features(d, card)
    mw = np.sort(model.predict([P, F], verbose=0, batch_size=1024) * card["load_std"] + card["load_mean"], -1)
    low, mid, high = mw[..., 0], mw[..., 1], mw[..., 2]
    return mid - widen * (mid - low), mid, mid + widen * (high - mid)


def same_hour_yesterday(d):
    target = d["T"].tz_convert(None).values[:, None] + (np.arange(1, H + 1) * np.timedelta64(1, "h"))[None, :]
    return df["load_mw"].reindex(pd.DatetimeIndex(target.ravel(), tz="UTC") - pd.Timedelta(hours=24)).values.reshape(-1, H)


def score(name, d, low, mid, high):
    actual = d["y"]
    s = crps(actual, low, mid, high)
    return {"forecast": name, "hours": actual.size, "avg miss (MW)": round(float(np.abs(actual - mid).mean()), 1),
            "inside (%), aim 80": round(100 * float(((actual >= low) & (actual <= high)).mean()), 1),
            "width (MW)": round(float((high - low).mean())), "CRPS (MW)": round(float(s.mean()), 1),
            "CRPS skill": round(crps_skill(s, np.abs(same_hour_yesterday(d) - actual)), 3)}

## Part 5 · Validate on 2025 and pick `range_widen`
Train on 2023-2024 only, forecast every hour of 2025 with the HRRR run that was really available, and pick the widening
factor with the **lowest CRPS among those with at least 80% inside** (the FP2 rule).

In [ ]:
U = lambda s: pd.Timestamp(s, tz="UTC")
model_2024, card_2024 = train_ensemble(U("2025-01-01"))
valid = subset(newest_run, U("2025-01-01"), U("2026-01-01"))
low, mid, high = forecast(model_2024, card_2024, valid)
factors = np.round(np.arange(0.9, 1.81, 0.05), 2)
rows = []
for f in factors:
    lo, hi = mid - f * (mid - low), mid + f * (high - mid)
    rows.append(score(f"widen {f}", valid, lo, mid, hi))
table = pd.DataFrame(rows)
ok = table[table["inside (%), aim 80"] >= 80]
best_widen = float(factors[ok["CRPS (MW)"].idxmin()]) if len(ok) else float(factors[-1])
print("chosen range_widen:", best_widen)
yesterday_2025 = same_hour_yesterday(valid)
pd.DataFrame([score("same hour yesterday", valid, yesterday_2025, yesterday_2025, yesterday_2025),
              score("ensemble, no widening", valid, low, mid, high),
              table.loc[list(factors).index(best_widen)].to_dict()])

## Part 6 · The one look at 2026 (the holdout)
Train again on 2023-**2025** (the same recipe, nothing re-tuned) and forecast Jan to Oct 2026 with the HRRR runs as
issued and the `range_widen` from Part 5. This is the honest number for the report and the card's test-year MAE.

In [ ]:
model_2025, card_2025 = train_ensemble(U("2026-01-01"))
test = subset(newest_run, U("2026-01-01"), U("2030-01-01"))
low, mid, high = forecast(model_2025, card_2025, test, best_widen)
yesterday_2026 = same_hour_yesterday(test)
test_rows = pd.DataFrame([score("same hour yesterday", test, yesterday_2026, yesterday_2026, yesterday_2026),
                          score(f"ensemble, widen {best_widen}", test, low, mid, high)])
test_mae = float(np.abs(test["y"] - mid).mean())
recent = subset(newest_run, U("2026-08-15"), U("2026-10-17"))
lr, mr, hr_ = forecast(model_2025, card_2025, recent, best_widen)
test_rows = pd.concat([test_rows, pd.DataFrame([score("ensemble, Aug 15 - Oct 15 2026", recent, lr, mr, hr_)])])
test_rows

In [ ]:
# The forecast issued at 6 AM each day of the hottest week of 2026, with its range
local = test["T"].tz_convert("America/New_York")
hot = df.loc["2026-01-01":, "temp_f"].idxmax().tz_convert("America/New_York").normalize()
plt.figure(figsize=(12, 4))
for k in np.where((local.hour == 6) & (np.abs((local.normalize() - hot).days) <= 3))[0]:
    hrs = pd.date_range(test["T"][k] + pd.Timedelta(hours=1), periods=H, freq="h").tz_convert("America/New_York")
    plt.fill_between(hrs, low[k], high[k], color="tab:blue", alpha=0.3)
    plt.plot(hrs, mid[k], color="tab:blue", linewidth=1)
    plt.plot(hrs, test["y"][k], color="black", linewidth=1.5)
plt.title("2026 holdout, hottest week: blue = 80% range and expected value (HRRR as issued), black = actual")
plt.ylabel("MW"); plt.tight_layout(); plt.show()

## Part 7 · The final fit and the model card
Same recipe on **every hour through Oct 2026** (the newest load shape), with the Part 5 `range_widen`.
`"weather": "hrrr"`: the cloud gives the model the newest HRRR run from the class file, like in training (if that file
is ever missing, the NWS forecast is used and sunlight is estimated from its cloud cover).

In [ ]:
model, card = train_ensemble(U("2030-01-01"))
card.update({"range_method": "lstm_quantiles", "quantiles": QUANTILES, "range_widen": best_widen, "weather": "hrrr",
             "members": SETTINGS["members"], "units": SETTINGS["units"], "epochs": SETTINGS["epochs"],
             "trained_through_utc": str(df.index.max()), "test_mae_mw": round(test_mae, 1),
             "test_2026": test_rows.to_dict("records"),
             "trained_at": pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%dT%H:%M:%SZ")})
model.save("model.keras")
with open("model_card.json", "w") as f:
    json.dump(card, f, indent=1)
reloaded = keras.models.load_model("model.keras", compile=False)
P, F, _ = features(test, card)
print("reloaded model reproduces the predictions:", bool(np.allclose(reloaded.predict([P[:16], F[:16]], verbose=0),
                                                                       model.predict([P[:16], F[:16]], verbose=0), atol=1e-4)))
print("saved model.keras + model_card.json | range_widen", best_widen, "| params", model.count_params())

## Part 8 · Ship it
**Once, before the first ship of this model:** my fork's `forecast-24h-dl` has `make_inputs_rama_v1` (the copy of
Part 3's `make_inputs`). On GitHub: Actions > "Deploy forecast-24h-dl" > Run workflow, and wait for green.

In [ ]:
PROJECT_ID = "energy-raa20002-f26-515102"
UPLOAD = False                          # True to ship
BUCKET_NAME = PROJECT_ID
if UPLOAD:
    from google.colab import auth
    from google.cloud import storage
    auth.authenticate_user()
    bucket = storage.Client(project=PROJECT_ID).bucket(BUCKET_NAME)
    bucket.blob("energy/models/model.keras").upload_from_filename("model.keras")
    bucket.blob("energy/models/model_card.json").upload_from_filename("model_card.json")
    stamp = card["trained_at"].replace(":", "-")
    bucket.blob("energy/models/history/model_" + stamp + ".keras").upload_from_filename("model.keras")
    bucket.blob("energy/models/history/model_card_" + stamp + ".json").upload_from_filename("model_card.json")
    print("shipped: gs://" + BUCKET_NAME + "/energy/models/model.keras")
    !gcloud scheduler jobs run forecast-24h-dl-hourly --location=us-central1 --project={PROJECT_ID}
else:
    print("UPLOAD is False: nothing shipped.")

In [ ]:
if UPLOAD:
    time.sleep(90)
    latest = pd.read_csv(io.BytesIO(bucket.blob("energy/forecasts/latest.csv").download_as_bytes()))
    print(latest[["target_hour_utc", "predicted_low_mw", "predicted_load_mw", "predicted_high_mw", "range_method"]].head(8))